# Task3 双均线策略回测 Notebook

本 Notebook 对双均线策略的核心流程进行白盒展示，包括：数据读取、均线计算、金叉/死叉信号生成、策略回测与绩效指标评估。

## 1. 核心概念

- **金叉**：短均线向上突破长均线，通常被视为买入信号。
- **死叉**：短均线向下跌破长均线，通常被视为卖出信号。
- **累计回报**：策略在整个样本区间内累计获得的总收益。
- **最大回撤（MDD）**：策略净值从高点回落到低点时的最大跌幅。
- **夏普比率（Sharpe Ratio）**：单位风险对应的超额收益水平。

In [ ]:
import sys
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

base_dir = Path.cwd().resolve().parent
sys.path.insert(0, str(base_dir))

from strategy_utils import StrategyParams, apply_dual_ma_strategy, load_price_data, summarize_strategy

raw_dir = base_dir / 'raw_data'
processed_dir = base_dir / 'processed_data'
figures_dir = base_dir / 'figures'
default_params = StrategyParams(short_window=5, long_window=15)
metrics_df = pd.read_csv(processed_dir / 'ma_strategy_metrics.csv')
metrics_df

## 2. 加载样例股票并计算双均线策略

这里以中国巨石为例，展示如何从原始 CSV 开始计算短均线、长均线和交易信号。

In [ ]:
jushi = load_price_data(raw_dir / 'china-jushi_600176.csv')
jushi_strategy = apply_dual_ma_strategy(jushi, default_params.short_window, default_params.long_window)
jushi_strategy[['date', 'close', 'ma_short', 'ma_long', 'buy_signal', 'sell_signal', 'position']].tail(12)

In [ ]:
jushi_summary = summarize_strategy(jushi_strategy, '中国巨石（600176）', default_params)
pd.Series(jushi_summary)

## 3. 图表展示

下面的图片展示了默认参数 `MA5/MA15` 下的交易信号图和策略净值图。

In [ ]:
display(Image(filename=str(figures_dir / 'china_jushi_ma_5_15_signals.png')))
display(Image(filename=str(figures_dir / 'china_jushi_ma_5_15_equity.png')))
display(Image(filename=str(figures_dir / 'zhongji_xuchuang_ma_5_15_signals.png')))
display(Image(filename=str(figures_dir / 'zhongji_xuchuang_ma_5_15_equity.png')))

## 4. 不同参数组合比较

为了观察均线周期变化对收益和风险的影响，本次额外测试了 `MA5/20` 与 `MA10/30` 两组参数。

In [ ]:
metrics_df[['stock', 'short_window', 'long_window', 'strategy_cumulative_return', 'max_drawdown', 'sharpe_ratio']]

In [ ]:
display(Image(filename=str(figures_dir / 'ma_strategy_metric_comparison.png')))

## 5. 观察总结

1. 双均线策略适合趋势明显的区间。
2. 中际旭创在样本期内弹性更强，因此策略收益更高，但回撤也更大。
3. `MA5/20` 在本次样本中整体表现最好，说明适度拉长长均线可以减少噪声干扰。